In [1]:
# import Pkg; Pkg.add(["DataStructures", "OnlineStats"])
using Agents
using Random
import StatsBase: middle
import StaticArraysCore: SVector, MVector
using FLoops
using DataStructures: OrderedDict
import LinearAlgebra: norm
using Makie, Makie.Observables
using OnlineStats
import DataStructures: DefaultDict
using ThreadPinning
using TimerOutputs

In [2]:
last_model_step_time::UInt64 = time_ns()
avg_model_step_duration::Observable{Mean{Float64}} = Observable(Mean(weight=HarmonicWeight(30)))

Observable(Mean: n=0 | value=0.0)
    0 => (sc::Observables.SetindexCallback)(x) @ Observables ~/.julia/packages/Observables/YdEbO/src/Observables.jl:148


In [3]:
abstract type ParticleColor end
struct Red <: ParticleColor end
struct Green <: ParticleColor end
struct Yellow <: ParticleColor end
struct Cyan <: ParticleColor end
struct Orange <: ParticleColor end

In [4]:
@agent struct Par(ContinuousAgent{2, Float64})
    color::ParticleColor
    # pos + vel are predifined in ContinuousAgent
end

@agent struct Particle(ContinuousAgent{2,Float64})
    const speed::Float64
    const cohere_factor::Float64
    const separation::Float64
    const separate_factor::Float64
    const match_factor::Float64
    const visual_distance::Float64
end

LoadError: LoadError: MethodError: no method matching var"@agent"(::LineNumberNode, ::Module, ::Expr)
The function `@agent` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  var"@agent"(::LineNumberNode, ::Module, ::Any, !Matched::Any, !Matched::Any)
   @ Agents ~/.julia/packages/Agents/xtlGn/src/core/agents.jl:210
  var"@agent"(::LineNumberNode, ::Module, ::Any, !Matched::Any, !Matched::Any, !Matched::Any)
   @ Agents ~/.julia/packages/Agents/xtlGn/src/core/agents.jl:172

in expression starting at /Users/isaiahday/Documents/Programming/particle-sim/particle-sim (julia)/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W3sZmlsZQ==.jl:1

In [5]:
function initialize_model(to=TimerOutput())
    extent::NTuple{2, Float64} = 3 .* (500.0, 500.0);
    space2d = ContinuousSpace(extent;
                              periodic=false,
                              # spacing=20,
                              );
    model = AgentBasedModel(Par, space2d;
                            agent_step! = (args...) -> (@timeit to "agent_step!" agent_step!(args...)),
                            model_step! = (args...) -> (@timeit to "model_step!" model_step!(args...; to=to)),
                            properties=push!(Dict{Symbol, Float64}(
                                    lhs_rhs=>rand(range)
                                    for (lhs_rhs, range) in properties),
                                :time_scale => 1.0)
                            )

    for c in [Red(), Green(), Orange(), Cyan()]
        for _ in 1:750
            vel =  SVector(0., 0.)
            add_agent!(model, vel, c)
        end
    end

    model
end


initialize_model (generic function with 2 methods)

In [6]:
color_interact(lhs::ParticleColor,  rhs::ParticleColor, m::ABM) =
    abmproperties(m)[Symbol(string(color_sym(lhs))*'_'*string(color_sym(rhs)))]

color_sym(p::Par) = color_sym(p.color)
color_sym(::ParticleColor) = :black
color_sym(::Green) = :green
color_sym(::Red) = :red
color_sym(::Orange) = :orange
color_sym(::Cyan) = :cyan
color_sym(::Yellow) = :yellow

properties=OrderedDict(
    :red_red       => -1:0.1:1,
    :red_green     => -1:0.1:1,
    :red_orange    => -1:0.1:1,
    :red_cyan      => -1:0.1:1,
    :green_red     => -1:0.1:1,
    :green_green   => -1:0.1:1,
    :green_orange  => -1:0.1:1,
    :green_cyan    => -1:0.1:1,
    :orange_red    => -1:0.1:1,
    :orange_green  => -1:0.1:1,
    :orange_orange => -1:0.1:1,
    :orange_cyan   => -1:0.1:1,
    :cyan_red      => -1:0.1:1,
    :cyan_green    => -1:0.1:1,
    :cyan_orange   => -1:0.1:1,
    :cyan_cyan     => -1:0.1:1,
    :viscosity     =>  0:.01:1,
)

UndefVarError: UndefVarError: `Par` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [7]:
agent_step!(agent, model) = move_agent!(agent, model, abmproperties(model)[:time_scale])

agent_step! (generic function with 1 method)

In [8]:
function model_step!(model; to=TimerOutput())
    @timeit to "update_vel! loop" begin
        # about 20% speedup to extract the viscosity var first.
        viscosity::Float64 = abmproperties(model)[:viscosity]
        @floop for agent in collect(allagents(model))
            update_vel!(agent, model; viscosity=viscosity)
        end
    end
    @timeit to "max reduction" begin
        max_vel = maximum(map(x->norm(x.vel), allagents(model)))
    end
    @timeit to "mean reduction" begin
        mean_vel = mean(map(x->norm(x.vel), allagents(model)))
    end
    # model.time_scale = max(0.1/max_vel, 1.)
    if max_vel*model.time_scale > getfield(model, :space).spacing || mean_vel*model.time_scale > 30
        model.time_scale /= 1.1
    end
    if model.time_scale < 0.9
        model.time_scale *= 1.01
    elseif model.time_scale > 1.1
        model.time_scale /= 1.01
    end
    @debug model.time_scale

    global last_model_step_time
    global avg_model_step_duration
    delta_time = time_ns() - last_model_step_time
    last_model_step_time = time_ns()
    fit!(avg_model_step_duration[], delta_time/1e9)
    notify(avg_model_step_duration)  # to update the fps label
end

model_step! (generic function with 1 method)

In [9]:
function update_vel!(agent::Par, model::ABM; viscosity::Union{Nothing, Float64}=nothing)
    force = sum(
        let g = color_interact(agent.color, other.color, model),
            d = euclidean_distance(agent, other, model)
            (0 < d < 80 ? (g / d .* (agent.pos - other.pos)) : zero(SVector{2, Float64}))
        end
        for other in Agents.nearby_agents(agent, model, 80);
        init = zero(SVector{2, Float64}),
    )
    # push away from border
    force += 0.1*(max.(40 .- agent.pos, 0)
                 - max.(40 .- (spacesize(model) - agent.pos), 0))

    # combine past velocity and current force
    viscosity = (isnothing(viscosity) ? abmproperties(model)[:viscosity] : viscosity)
    agent.vel = agent.vel * (1-viscosity) + force
end

UndefVarError: UndefVarError: `Par` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [10]:


function run_sim(; to=TimerOutput())
    # this is basically it, but we want to rearrange the layout a bit.
    model = make_model(to)
    fig, ax, abmobs = with_theme(theme_dark()) do
        abmplot(model;
                ac=color_sym, as=8.0,  # agent color and size
                params=ParticleLife.properties,
                scatterkwargs=(; :markerspace=>:data),
                enable_inspection=false)
    end

    # the rest is mostly changing the layout a bit and adding a randomize button and an fps label
    controls = content(fig[2,1][1,1])
    param_sliders = content(fig[2,1][1,2])
    update_button = content(param_sliders[2,1])

    # the size is currently fixed, probably not a good long-term solution
    ax.width[] = 1000
    ax.height[] = 1500
    ui = fig[1,2] = GridLayout();
    param_sliders.width[] = 500
    ui[1,1] = param_sliders
    sg::SliderGrid = content(param_sliders[1,1])
    update_and_rand_button = param_sliders[2,1] = GridLayout();
    update_and_rand_button[1,1] = update_button
    rand_button = Button(update_and_rand_button[1,2], label="randomize", tellwidth=true)
    on(rand_button.clicks) do _  # randomize params
        for s in sg.sliders
            set_close_to!(s, rand(s.range[]))
        end
        update_button.clicks[] += 1
        abmproperties(model)[:time_scale] = 1.0
    end
    Label(ui[2,1], "----------------------")
    ui[3,1] = controls
    Label(ui[4,1], "----------------------")
    # show fps
    empty!(avg_model_step_duration.listeners)
    fps = throttle(0.5, @lift 1/value($avg_model_step_duration))
    fps_label = Label(ui[5,1], text="0.0 fps")
    on(fps) do fps
        fps_label.text = "$(round(fps, digits=2)) fps"
    end

    Makie.deleterow!(fig.layout, 2)
    fig
end

run_sim (generic function with 1 method)

In [12]:
function custom_model_step!(model)
    scheduler = Schedulers
    for id in scheduler(model)
        agent_step!(model[id], model)
    end
    return model
end


custom_model_step! (generic function with 1 method)

In [13]:
using GLMakie
function make_video()
    GLMakie.activate!() # hide
    with_theme(theme_dark()) do
        abmvideo("foo.mp4", initialize_model(), ac=color_sym, as=8.0;
                scatterkwargs=(; :markerspace=>:data))
    end
end

make_video (generic function with 1 method)

In [14]:
make_video()

UndefVarError: UndefVarError: `properties` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing IterTools in the current active module Main

In [15]:
make_video()

UndefVarError: UndefVarError: `properties` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing IterTools in the current active module Main

In [16]:
make_video()

UndefVarError: UndefVarError: `properties` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing IterTools in the current active module Main